# Event Labeling

## Process the Data

- **Purpose:** Create triple-barrier direction labels from the integrated point-in-time feature schema.
- **Settings:** `return_horizon=1000` bars; `ewma_span=100`; `vertical_barrier=1000` bars; `pt_sl=(1.0, 1.0)`; `min_target=development 25th percentile`; `min_class_frequency=0.10`.
- **Data:** News-triggered candidates produce one labeled-event artifact with inline development and holdout metadata.
- **Decision:** Fit thresholds on development only, remove rare labels, and purge development events that cross the fixed holdout boundary.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parents[1]

from src.preprocessing.event_labeling import build_labeled_event_data

period = "2025-01-01_2025-12-31"
event_dir = PROJECT_ROOT / "data/research_data/events"
candidate_path = event_dir / f"aapl_event_candidates_{period}.parquet"
dollar_path = PROJECT_ROOT / f"data/research_data/market/features/aapl_dollar_bar_{period}.parquet"
labeled_path = event_dir / f"aapl_labeled_events_{period}.parquet"

candidate_split = pd.read_parquet(candidate_path).sort_values("event_start", ignore_index=True)
dollar_bars = pd.read_parquet(dollar_path).sort_values("end").drop_duplicates("end", keep="last")


In [2]:
labeled_events = build_labeled_event_data(candidate_split, dollar_bars)

event_dir.mkdir(parents=True, exist_ok=True)
labeled_events.to_parquet(labeled_path, index=False)
print(labeled_path)


/Users/kwonjunhyuk9/Documents/financial-machine-learning/data/research_data/events/aapl_labeled_events_2025-01-01_2025-12-31.parquet


## Take a Quick Look at the Data Structure

- **Purpose:** Inspect development class balance, numeric ranges, and the persisted event schema.
- **Settings:** No analytical parameters; read-only inspection.
- **Data:** Inspect only the development rows of the persisted 62-column event artifact.
- **Decision:** Leave labels, holdout outcomes, and the artifact unchanged.

In [3]:
development_data = labeled_events.loc[
    labeled_events["partition"].eq("development")
]
development_data.head()


,event_start,symbol,event_end,vertical_barrier,target_return,raw_return,direction_label,partition,holdout_boundary,mean_sentiment_score,...,Bollinger Band Middle,Bollinger Band Lower,True Range,Average True Range,Keltner Channel Upper,Keltner Channel Middle,Keltner Channel Lower,Donchian Channel Upper,Donchian Channel Middle,Donchian Channel Lower
0,2025-01-13 14:30:01.329809+00:00,AAPL,2025-01-13 14:34:17.401727+00:00,2025-01-16 18:19:25.547320+00:00,0.006302,-0.006778,-1,development,2025-10-16 17:14:45.827270+00:00,0.796031,...,241.9314,236.8492,9.60,0.8168,242.9775,241.3440,239.7104,242.800,237.9350,233.070
1,2025-01-16 14:30:01.488226+00:00,AAPL,2025-01-16 14:34:43.996829+00:00,2025-01-21 17:48:33.046371+00:00,0.004690,-0.004820,-1,development,2025-10-16 17:14:45.827270+00:00,0.168639,...,237.7111,237.5796,0.42,0.1261,237.9677,237.7155,237.4634,237.840,237.6300,237.420
2,2025-01-17 14:30:01.792073+00:00,AAPL,2025-01-17 14:39:01.069235+00:00,2025-01-22 20:42:17.531895+00:00,0.006821,-0.007032,-1,development,2025-10-16 17:14:45.827270+00:00,-0.898147,...,228.4782,226.5652,3.92,0.3121,229.3303,228.7060,228.0818,232.115,230.1125,228.110
3,2025-01-17 17:20:53.298606+00:00,AAPL,2025-01-21 14:30:00.854179+00:00,2025-01-23 16:19:31.331783+00:00,0.005790,-0.025297,-1,development,2025-10-16 17:14:45.827270+00:00,0.013412,...,229.7761,229.4412,0.17,0.2114,230.2595,229.8367,229.4138,230.110,229.8025,229.495
4,2025-01-21 14:30:00.854179+00:00,AAPL,2025-01-21 14:34:53.478136+00:00,2025-01-23 19:56:48.170064+00:00,0.005329,-0.007594,-1,development,2025-10-16 17:14:45.827270+00:00,-0.456326,...,229.3857,226.1968,6.15,0.5650,230.1386,229.0086,227.8786,230.010,226.9350,223.860


In [4]:
development_data.info()


<class 'pandas.DataFrame'>
RangeIndex: 176 entries, 0 to 175
Data columns (total 62 columns):
 #   Column                                    Non-Null Count  Dtype              
---  ------                                    --------------  -----              
 0   event_start                               176 non-null    datetime64[us, UTC]
 1   symbol                                    176 non-null    str                
 2   event_end                                 176 non-null    datetime64[us, UTC]
 3   vertical_barrier                          176 non-null    datetime64[us, UTC]
 4   target_return                             176 non-null    float64            
 5   raw_return                                176 non-null    float64            
 6   direction_label                           176 non-null    int8               
 7   partition                                 176 non-null    str                
 8   holdout_boundary                          176 non-null    datetime64[us

In [5]:
development_data["direction_label"].value_counts()


direction_label
 1    95
-1    81
Name: count, dtype: int64

In [6]:
development_data.select_dtypes(include="number").describe()


,target_return,raw_return,direction_label,mean_sentiment_score,fractionally_differenced_log_close,McClellan Oscillator,Advancers - Decliners,On-Balance Volume,Accumulation/Distribution Line,Chaikin Oscillator,...,Bollinger Band Middle,Bollinger Band Lower,True Range,Average True Range,Keltner Channel Upper,Keltner Channel Middle,Keltner Channel Lower,Donchian Channel Upper,Donchian Channel Middle,Donchian Channel Lower
count,176.000000,176.000000,176.000000,176.000000,176.000000,176.000000,176.000000,1.760000e+02,1.760000e+02,176.000000,...,176.000000,176.000000,176.000000,176.000000,176.000000,176.000000,176.000000,176.000000,176.000000,176.000000
mean,0.007325,0.001381,0.079545,0.062438,1.995935,0.158105,216.018778,1.303766e+06,2.231990e+06,-494.261978,...,216.317260,215.418034,1.289602,0.312129,216.924373,216.300121,215.675871,217.215114,216.204929,215.194744
std,0.003453,0.014146,0.999675,0.532535,0.029486,2.906571,16.432353,4.192176e+05,9.415255e+05,3781.131330,...,16.469311,16.467168,2.507197,0.197739,16.476464,16.456394,16.445817,16.584209,16.460019,16.434644
min,0.003851,-0.074197,-1.000000,-0.964634,1.909065,-9.475200,171.930000,1.165390e+05,1.994439e+05,-10572.948200,...,172.535400,171.810900,0.080000,0.077100,173.579300,172.547900,171.516400,173.460000,171.040000,168.620000
25%,0.004679,-0.006470,-1.000000,-0.308549,1.974659,-0.334700,203.316250,1.041767e+06,1.887511e+06,-2827.743550,...,203.450050,203.062850,0.193750,0.207900,204.035850,203.464000,202.966825,204.505000,203.742500,202.975000
50%,0.006228,0.004508,1.000000,0.051556,1.989623,0.436200,211.250000,1.267749e+06,2.097338e+06,-437.022000,...,211.167150,210.492800,0.355000,0.260350,211.896850,211.276100,210.727000,212.065000,210.997500,210.247500
75%,0.009311,0.007448,1.000000,0.496559,2.021439,1.569200,230.522500,1.499774e+06,2.806536e+06,2614.949450,...,230.913050,229.952225,1.120000,0.343025,231.379925,230.847025,230.314200,231.758750,231.005625,229.747500
max,0.023920,0.078197,1.000000,0.941092,2.061100,14.341000,254.570000,2.336261e+06,4.134200e+06,9462.076500,...,257.003200,255.587500,17.560000,1.415700,257.632900,256.902200,256.171500,257.430000,255.610000,253.790000


In [7]:
development_data.select_dtypes(include="number").replace([np.inf, -np.inf], np.nan).hist(figsize=(20, 24), bins=30)


array([[<Axes: title={'center': 'target_return'}>,
        <Axes: title={'center': 'raw_return'}>,
        <Axes: title={'center': 'direction_label'}>,
        <Axes: title={'center': 'mean_sentiment_score'}>,
        <Axes: title={'center': 'fractionally_differenced_log_close'}>,
        <Axes: title={'center': 'McClellan Oscillator'}>,
        <Axes: title={'center': 'Advancers - Decliners'}>],
       [<Axes: title={'center': 'On-Balance Volume'}>,
        <Axes: title={'center': 'Accumulation/Distribution Line'}>,
        <Axes: title={'center': 'Chaikin Oscillator'}>,
        <Axes: title={'center': 'New Highs - New Lows'}>,
        <Axes: title={'center': 'Money Flow Index'}>,
        <Axes: title={'center': 'Williams %R'}>,
        <Axes: title={'center': 'Aroon Indicator Up'}>],
       [<Axes: title={'center': 'Aroon Indicator Down'}>,
        <Axes: title={'center': 'Commodity Channel Index'}>,
        <Axes: title={'center': 'Relative Vigor Index'}>,
        <Axes: title={'cen